In [8]:
import pandas as pd
import numpy as np

In [9]:
df_de = pd.read_csv(
    "../data/time_series.csv"
)

In [10]:
df_de["datetime"] = pd.to_datetime(
    df_de["utc_timestamp"]
).dt.tz_localize(None)

C:\Users\panos\AppData\Local\Temp\ipykernel_24580\2479405543.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_de["datetime"] = pd.to_datetime(


In [11]:
weather = pd.read_csv(
    "../data/weather_germany.csv"
)

In [12]:
weather["datetime"] = pd.to_datetime(
    weather["datetime"]
)

In [13]:
weather["datetime"].dtype

dtype('<M8[us]')

In [14]:
df_de["datetime"].dtype

dtype('<M8[us]')

In [15]:
df_de = df_de[[
    "datetime",
    "DE_load_actual_entsoe_transparency",
    "DE_solar_generation_actual",
    "DE_wind_onshore_generation_actual",
    "DE_wind_offshore_generation_actual"
]]

In [16]:
df_de.rename(
    columns={
        "DE_load_actual_entsoe_transparency": "load",
        "DE_solar_generation_actual": "solar",
        "DE_wind_onshore_generation_actual": "wind_onshore",
        "DE_wind_offshore_generation_actual": "wind_offshore"
    },
    inplace=True
)

In [17]:
df_final = df_de.merge(
    weather,
    on="datetime",
    how="inner"
)

In [18]:
df_final.shape

(50400, 9)

In [20]:
df_final.to_csv(
    "../data/germany_energy_weather.csv",
    index=False
)

In [23]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 50400 entries, 0 to 50399
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   datetime         50400 non-null  datetime64[us]
 1   load             50400 non-null  float64       
 2   solar            50297 non-null  float64       
 3   wind_onshore     50328 non-null  float64       
 4   wind_offshore    50326 non-null  float64       
 5   temperature      50400 non-null  float64       
 6   wind_speed       50400 non-null  float64       
 7   cloud_cover      50400 non-null  int64         
 8   solar_radiation  50400 non-null  float64       
dtypes: datetime64[us](1), float64(7), int64(1)
memory usage: 3.5 MB


In [24]:
df_final = df_final.sort_values("datetime")


df_final.interpolate(
    method="linear",
    inplace=True
)

,datetime,load,solar,wind_onshore,wind_offshore,temperature,wind_speed,cloud_cover,solar_radiation
0,2015-01-01 00:00:00,41151.0,NaN,8336.0,517.0,3.8,14.4,71,0.0
1,2015-01-01 01:00:00,40135.0,NaN,8540.0,514.0,3.6,14.9,69,0.0
2,2015-01-01 02:00:00,39106.0,NaN,8552.0,518.0,3.3,14.6,95,0.0
3,2015-01-01 03:00:00,38765.0,NaN,8643.0,520.0,3.0,14.1,82,0.0
4,2015-01-01 04:00:00,38941.0,NaN,8712.0,520.0,2.3,13.0,67,0.0
...,...,...,...,...,...,...,...,...,...
50395,2020-09-30 19:00:00,57559.0,0.0,5900.0,4754.0,11.8,7.6,45,0.0
50396,2020-09-30 20:00:00,54108.0,0.0,6642.0,5194.0,11.2,7.7,9,0.0
50397,2020-09-30 21:00:00,49845.0,0.0,6829.0,5339.0,11.2,7.0,14,0.0
50398,2020-09-30 22:00:00,46886.0,0.0,7134.0,5399.0,11.0,6.6,7,0.0


In [25]:
df_final.isnull().sum()

datetime           0
load               0
solar              7
wind_onshore       0
wind_offshore      0
temperature        0
wind_speed         0
cloud_cover        0
solar_radiation    0
dtype: int64

In [26]:
df_final["solar"].describe()

count    50393.000000
mean      4557.344472
std       6936.511726
min          0.000000
25%          0.000000
50%        164.000000
75%       7315.000000
max      32947.000000
Name: solar, dtype: float64

In [27]:
df_final[df_final["solar"].isnull()]

,datetime,load,solar,wind_onshore,wind_offshore,temperature,wind_speed,cloud_cover,solar_radiation
0,2015-01-01 00:00:00,41151.0,NaN,8336.0,517.0,3.8,14.4,71,0.0
1,2015-01-01 01:00:00,40135.0,NaN,8540.0,514.0,3.6,14.9,69,0.0
2,2015-01-01 02:00:00,39106.0,NaN,8552.0,518.0,3.3,14.6,95,0.0
3,2015-01-01 03:00:00,38765.0,NaN,8643.0,520.0,3.0,14.1,82,0.0
4,2015-01-01 04:00:00,38941.0,NaN,8712.0,520.0,2.3,13.0,67,0.0
5,2015-01-01 05:00:00,39045.0,NaN,9167.0,521.0,1.6,12.3,81,0.0
6,2015-01-01 06:00:00,40206.0,NaN,9811.0,520.0,0.9,14.8,84,0.0


In [28]:
df_final["solar"] = df_final["solar"].fillna(0)

In [29]:
df_final.isnull().sum()

datetime           0
load               0
solar              0
wind_onshore       0
wind_offshore      0
temperature        0
wind_speed         0
cloud_cover        0
solar_radiation    0
dtype: int64

In [31]:
df_final["wind_onshore"] = (
    df_final["wind_onshore"]
    .interpolate()
)

df_final["wind_offshore"] = (
    df_final["wind_offshore"]
    .interpolate()
)

In [32]:
df_final.to_csv(
    "../data/germany_energy_weather.csv",
    index=False
)

In [33]:
df_final.describe()

,datetime,load,solar,wind_onshore,wind_offshore,temperature,wind_speed,cloud_cover,solar_radiation
count,50400,50400.000000,50400.000000,50400.000000,50400.000000,50400.000000,50400.000000,50400.000000,50400.000000
mean,2017-11-15 23:30:00,55492.468552,4556.711508,9582.989841,1971.107560,11.030500,13.355125,64.506111,130.894722
min,2015-01-01 00:00:00,31307.000000,0.000000,119.000000,0.000000,-12.800000,0.000000,0.000000,0.000000
25%,2016-06-08 23:45:00,47106.000000,0.000000,3539.000000,587.000000,4.600000,8.500000,29.000000,0.000000
50%,2017-11-15 23:30:00,55092.000000,164.000000,7133.000000,1642.000000,10.500000,12.400000,82.000000,7.000000
75%,2019-04-24 23:15:00,64309.250000,7315.000000,13305.500000,3066.000000,17.200000,17.100000,100.000000,201.000000
max,2020-09-30 23:00:00,77549.000000,32947.000000,40752.000000,6901.000000,37.400000,60.200000,100.000000,882.000000
std,NaN,10015.431042,6936.237916,7984.872473,1566.698034,8.221422,6.365118,37.759441,200.951961
